# Week 12 Demo: Combining and Reshaping DataFrames

## How to Work Through This Demo

Before every code block, read the setup text that explains what the code is about to do. After running the cell, read the explanation that follows to understand what the output shows. Run cells in order from top to bottom. Earlier cells create variables that later cells depend on.

If any cell produces an error, check that you ran all the cells above it first. Use Restart Kernel and Run All to reset if needed.

---

## Introduction

Every DataFrame operation you have used in this course (loading, cleaning, filtering, aggregating) has worked with a single table. That is often not how real data arrives. A database might store entity records in one table and event records in another. A data export might produce separate files for different time periods or geographic regions. A reporting system might deliver measurements in a wide format where each column is a category rather than each row being a single observation.

Chapter 8 introduces the tools pandas provides to work with data spread across multiple sources: `pd.concat()` for stacking DataFrames that share the same structure, `pd.merge()` for joining DataFrames on shared keys, and `pd.melt()` for reshaping wide data into long format.

If you have taken database modeling, `pd.merge()` will look familiar. It implements the same join logic as SQL: matching rows from two tables based on a shared key column. The `how` parameter controls the join type (inner, left, right, or outer) and maps directly to the SQL JOIN types you already know.

---

## Setup

Before running any cells, download the following files from the course page and place them in your `week12` folder alongside this notebook:

- `entity_registry.csv`
- `incident_reports_north.csv`
- `incident_reports_south.csv`
- `monthly_sightings.csv`

In [ ]:
import pandas as pd

---

## Part 1: Stacking DataFrames with `pd.concat()`

### The scenario

The Bureau of Paranormal Investigation maintains two regional offices. The Northern Bureau and the Southern Bureau each export their incident reports as separate CSV files. Before any analysis can run, these two files need to be combined into a single working table.

This is the simplest case for combining DataFrames: two tables with identical column structure that need to be stacked on top of each other.

### Load the regional files

In [ ]:
north = pd.read_csv("incident_reports_north.csv")
south = pd.read_csv("incident_reports_south.csv")

print("Northern Bureau:")
print(north.shape)
print(north.head(3))

In [ ]:
print("Southern Bureau:")
print(south.shape)
print(south.head(3))

**Understanding the output:**

Both DataFrames have the same seven columns: `incident_id`, `entity_id`, `location`, `date`, `investigator`, `anomaly_severity`, and `containment_status`. The Northern Bureau has 15 incidents; the Southern Bureau has 12.

Notice that each DataFrame has its own default index starting at 0. When you stack them, pandas needs to decide what to do with those indexes.

### Concatenate with `pd.concat()`

In [ ]:
all_incidents = pd.concat([north, south], ignore_index=True)

print(f"Combined: {all_incidents.shape}")
print(all_incidents.head(3))
print("...")
print(all_incidents.tail(3))

**Understanding `pd.concat()`:**

`pd.concat()` takes a list of DataFrames and stacks them along an axis. The default behavior stacks rows (`axis="index"`), which is what we want here.

The `ignore_index=True` parameter tells pandas to discard the original indexes and assign a new sequential index starting at 0. Without it, the index from `north` (0 through 14) would be followed by the index from `south` (also 0 through 11), producing duplicate index values. That is usually not what you want when combining records from different sources.

After concatenation: 15 + 12 = 27 rows, one clean sequential index from 0 to 26.

### Verify the result

In [ ]:
print(all_incidents.info())
print("\nMissing values:")
print(all_incidents.isnull().sum())

**Understanding the output:**

All 27 rows are present, all columns are non-null, and the data types are what we expect. The combined DataFrame is ready to use.

`pd.concat()` also works along columns (`axis="columns"`) and with more than two DataFrames. Pass any number of DataFrames in the list. Chapter 8 covers the `keys` parameter, which labels each row with its source, creating a hierarchical index in the result.

---

## Part 2: Joining DataFrames with `pd.merge()`

### The scenario

The Bureau keeps a central registry of all known entities: their classification, origin dimension, and threat level. The incident reports reference entities by ID but do not include those details directly. To answer questions like "which classification of entity generates the most incidents?" or "what is the average threat level of entities that have been contained?", the incident data needs to be joined to the registry.

This is a **many-to-one** join: many incidents can reference the same entity, but each entity appears only once in the registry.

### Load the entity registry

In [ ]:
registry = pd.read_csv("entity_registry.csv")
print(registry)

**Understanding the output:**

The registry has 10 entities. Each has an `entity_id` that also appears in the incident reports. `entity_id` is the shared key that connects the two tables.

### Inner join: keep only matching rows

In [ ]:
merged_inner = pd.merge(all_incidents, registry, on="entity_id")

print(f"Incidents: {len(all_incidents)} rows")
print(f"Registry: {len(registry)} rows")
print(f"Inner join result: {len(merged_inner)} rows")
print(f"\nEntities in result: {sorted(merged_inner['entity_name'].unique())}")

**Understanding the inner join:**

The `on="entity_id"` parameter tells pandas which column to use as the join key. An inner join keeps only rows where the key value appears in **both** tables. Every incident references an entity, so all 27 incident rows survive. But look at the entity count: only 8 of the 10 entities in the registry appear in the result.

To confirm which entities were dropped, compare the entity IDs in each table directly.

In [ ]:
registry_entities = set(registry["entity_id"])
incident_entities = set(all_incidents["entity_id"])
missing = registry_entities - incident_entities
print(f"Entities in registry but not in incidents: {missing}")
print(registry[registry["entity_id"].isin(missing)][["entity_id", "entity_name"]])

**Understanding the output:**

Yog-Sothoth and Shub-Niggurath are in the registry but have no incident reports. The inner join silently dropped them. Whether that is the right behavior depends on what you are analyzing. If you want a complete picture of all entities regardless of incident history, the inner join gives you the wrong answer.

### Left join: keep all rows from the left table

In [ ]:
merged_left = pd.merge(registry, all_incidents, on="entity_id", how="left")

print(f"Left join result: {len(merged_left)} rows")
print(f"\nRows where incident_id is NaN (no incidents):")
print(merged_left[merged_left["incident_id"].isna()][["entity_id", "entity_name", "classification", "threat_level"]])

**Understanding the left join:**

`how="left"` keeps every row from the **left** DataFrame (the registry) regardless of whether a match exists in the right DataFrame (the incidents). When no match is found, pandas fills the incident columns with `NaN`.

The result has 29 rows. That is 27 matched incident rows plus 2 unmatched registry rows for Yog-Sothoth and Shub-Niggurath. These are the Ancient Ones with no recorded incidents. Their absence is now visible rather than silently dropped.

Which join is "right" depends on the question. If you want to count incidents per entity and you need the zero counts to appear, use a left join on the registry. If you only care about entities that have incidents, use an inner join on the incident table.

### Right join: keep all rows from the right table

In [ ]:
merged_right = pd.merge(registry, all_incidents, on="entity_id", how="right")

print(f"Right join result: {len(merged_right)} rows")

**Understanding the right join:**

`how="right"` is the mirror of `how="left"`. Every row from the **right** DataFrame is kept. Unmatched rows from the left get `NaN`. Here the right table is `all_incidents`, so all 27 incident rows appear. This produces the same result as the inner join in this dataset because every incident references a valid entity. A right join becomes meaningful when the right table has rows with no match in the left, for example if an incident referenced an entity ID not yet in the registry.

### Outer join: keep all rows from both tables

In [ ]:
merged_outer = pd.merge(registry, all_incidents, on="entity_id", how="outer")

print(f"Outer join result: {len(merged_outer)} rows")
print(f"\nRows with NaN in incident_id (entities with no incidents):")
print(merged_outer[merged_outer["incident_id"].isna()][["entity_id", "entity_name"]])

**Understanding the outer join:**

`how="outer"` takes the union: every row from both tables appears in the result. Rows with no match on either side get `NaN` in the columns from the other table. This is the most inclusive option. Use it when you need a complete picture of everything in both tables.

### A note on key column names

When the key column has the same name in both DataFrames, `on="column_name"` works. When the column names differ, use `left_on=` and `right_on=` to specify each one separately. Chapter 8 covers merging on index values using `left_index=True` and `right_index=True`.

---

## Part 3: Reshaping with `pd.melt()`

### The scenario

The Bureau also tracks monthly sighting counts for each entity. The data arrived in **wide format**: one row per entity, with each month as its own column.

In [ ]:
sightings = pd.read_csv("monthly_sightings.csv")
print(sightings)

**Understanding wide format:**

Wide format is easy to read as a table and intuitive for data entry, but it is difficult to work with analytically. If you want to find the highest sighting count across all entities and months, you cannot filter a single column. The data is spread across six. If you want to sort by month, there is no month column to sort on.

**Long format** restructures this so each row is one observation: one entity, one month, one count. That gives you a single `month` column and a single `sighting_count` column, which makes every filtering and sorting operation straightforward.

### Reshape from wide to long with `pd.melt()`

In [ ]:
sightings_long = pd.melt(
    sightings,
    id_vars=["entity_id", "entity_name"],
    var_name="month",
    value_name="sighting_count"
)

print(f"Wide format: {sightings.shape}")
print(f"Long format: {sightings_long.shape}")
print(sightings_long.head(12))

**Understanding `pd.melt()`:**

`pd.melt()` takes a wide DataFrame and converts it to long format.

- `id_vars` specifies the columns that stay as identifiers. These repeat for each observation. Here `entity_id` and `entity_name` identify which entity each row belongs to.
- `var_name` gives a name to the new column that holds the former column headers. The month names (`Jan`, `Feb`, etc.) become values in a column called `month`.
- `value_name` gives a name to the new column that holds the values. The sighting counts become values in a column called `sighting_count`.

The result has 10 entities × 6 months = 60 rows. Every observation is now a single row with a clear entity, month, and count.

### Analyze the long format data

In [ ]:
# Which entity had the highest sighting count in any single month?
top = sightings_long.sort_values("sighting_count", ascending=False).head(5)
print("Top 5 single-month sighting counts:")
print(top[["entity_name", "month", "sighting_count"]])

In [ ]:
# Filter to a single month
may_sightings = sightings_long[sightings_long["month"] == "May"].sort_values(
    "sighting_count", ascending=False
)
print("\nMay sightings by entity:")
print(may_sightings[["entity_name", "sighting_count"]])

**Understanding the output:**

Both of these operations would have been awkward in the wide format. In long format, they are standard single-column operations. This is why reshaping data before analysis matters. The format determines what operations are natural.

---

## Conclusion

### What You've Learned

This demo introduced three tools for working with data spread across multiple sources.

**`pd.concat()`** stacks DataFrames that share the same structure. Use `ignore_index=True` when the row index carries no meaningful information and you want a clean sequential index in the result. Pass any number of DataFrames in a list.

**`pd.merge()`** joins two DataFrames on a shared key column, the same logic as a SQL JOIN. The `how` parameter controls which rows survive: `"inner"` keeps only matching rows, `"left"` keeps all rows from the left table, `"right"` keeps all rows from the right table, and `"outer"` keeps all rows from both. The choice of join type is a data question, not a technical one. It depends on what you need to know and whether missing matches should appear as `NaN` or be dropped entirely.

**`pd.melt()`** reshapes wide data into long format. `id_vars` identifies which columns stay as identifiers, `var_name` names the new column holding the former column headers, and `value_name` names the new column holding the values. Long format makes filtering, sorting, and grouping operations straightforward because each observation is a single row.

**`set_index()` and `reset_index()`** move columns in and out of the index. `set_index()` promotes a column to the index; `reset_index()` moves the index back into a column and restores the default integer index. They appear frequently as supporting operations alongside merges and reshapes.

### What the Textbook Covers Next

Chapter 8 goes further than this demo on several topics worth reading.

**Hierarchical indexing and `MultiIndex`:** pandas supports multiple levels of row or column labels. A `MultiIndex` allows you to represent higher-dimensional data in a flat table. The `swaplevel()` and `sort_index(level=)` methods manipulate level order. This is the foundation for `stack()` and `unstack()`.

**`stack()` and `unstack()`:** `stack()` rotates column labels into row labels, producing a Series from a DataFrame. `unstack()` does the reverse. These are the lower-level operations that `pivot()` and `melt()` build on.

**`pivot()`:** The inverse of `melt()`. Converts long format back to wide format using one column as the row index, one as the column headers, and one as the values.

**`combine_first()`:** Patches missing values in one DataFrame with values from another at matching index positions. Useful for filling gaps from a secondary source without a full merge.

### Applying These Tools in Week 13

Week 13 is a cumulative application week. You will work with a multi-source dataset that requires you to combine, join, and reshape data before analysis can begin. The workflow follows the same pattern you have built across prior application weeks: assess the data, prepare it, then analyze it. The Chapter 8 tools are the preparation step for data that starts in multiple files or the wrong shape.